# Content

1. **Import relevant libraries**
2. **Data inspection & handling before modification**
3. **Data audit & imputing missing values using scaffolding**
4. **Feature creation & data cleaning and modification**
5. **Preparing dataset for machine-learning model**

________________________________________________________________________
**1. Import relevent libraries**
________________________________________________________________________

In [11]:
# eCommerce and logistic project Feature engineering code

import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# loading the csv file for preprocessing

df = pd.read_csv("BADS_WS1920_known.csv")
df=pd.DataFrame(df)

print(df)

# Count missing values and Invalid dates in every column.

missing_by_column = df.isna().sum()

print("Missing or invalid values by column:")
print(missing_by_column.to_string())

print(f"\nTotal missing/invalid values across all columns: "
      f"{missing_by_column.sum()}")
display(df.head(10))
      

       order_item_id  order_date delivery_date  item_id item_size item_color  \
0                  1  2016-06-22    2016-06-27      643        38       navy   
1                  2  2016-06-22           NaN      337       152       grey   
2                  3  2016-06-22    2016-06-27      270       xxl       grey   
3                  4  2016-06-22    2016-06-27      142       xxl       grey   
4                  5  2016-06-22    2016-06-27      561       xxl       grey   
...              ...         ...           ...      ...       ...        ...   
99995          99996  2016-09-11    2016-09-12      156        20       blue   
99996          99997  2016-09-11    2016-09-12      156        20      brown   
99997          99998  2016-09-11    1994-12-31     1832       xxl      black   
99998          99999  2016-09-11    1994-12-31     1832       xxl      black   
99999         100000  2016-09-11    2016-10-08     1891        28    fuchsia   

       brand_id  item_price  user_id us

,order_item_id,order_date,delivery_date,item_id,item_size,item_color,brand_id,item_price,user_id,user_title,user_dob,user_state,user_reg_date,return
0,1,2016-06-22,2016-06-27,643,38,navy,30,49.90,30822,Mrs,1969-04-17,Saxony,2016-06-23,0
1,2,2016-06-22,NaN,337,152,grey,30,19.95,30822,Mrs,1969-04-17,Saxony,2016-06-23,0
2,3,2016-06-22,2016-06-27,270,xxl,grey,49,79.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
3,4,2016-06-22,2016-06-27,142,xxl,grey,49,99.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,0
4,5,2016-06-22,2016-06-27,561,xxl,grey,3,14.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
5,6,2016-06-22,2016-06-27,579,xxl,grey,3,19.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
6,7,2016-06-22,2016-06-27,72,41,grey,1,119.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,0
7,8,2016-06-22,2016-06-27,106,50,white,6,39.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
8,9,2016-06-22,2016-06-27,195,xxl,blue,46,13.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
9,10,2016-06-22,2016-06-27,195,xxl,grey,46,19.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1


___________________________________________________________________________________________

**2. Data inspection and handling before modification**
___________________________________________________________________________________________

In [12]:

# Convert date columns to correct standard date formats

date_columns = [
    "order_date",
    "delivery_date",
    "user_dob",
    "user_reg_date"]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

# Print conversion outcomes for easy reading as follows:

print("\nDate conversion outcomes:")

for column in date_columns:
    if column in df.columns:
        print(f"\n{column}:")
        print("Data type:", df[column].dtype)
        print("Valid dates:", df[column].notna().sum())
        print("Invalid or missing dates:", df[column].isna().sum())
        print(df[column].head())
    else:
        print(f"\n{column}: column not found in dataset")


Date conversion outcomes:

order_date:
Data type: datetime64[us]
Valid dates: 100000
Invalid or missing dates: 0
0   2016-06-22
1   2016-06-22
2   2016-06-22
3   2016-06-22
4   2016-06-22
Name: order_date, dtype: datetime64[us]

delivery_date:
Data type: datetime64[us]
Valid dates: 90682
Invalid or missing dates: 9318
0   2016-06-27
1          NaT
2   2016-06-27
3   2016-06-27
4   2016-06-27
Name: delivery_date, dtype: datetime64[us]

user_dob:
Data type: datetime64[us]
Valid dates: 91275
Invalid or missing dates: 8725
0   1969-04-17
1   1969-04-17
2   1970-04-22
3   1970-04-22
4   1970-04-22
Name: user_dob, dtype: datetime64[us]

user_reg_date:
Data type: datetime64[us]
Valid dates: 100000
Invalid or missing dates: 0
0   2016-06-23
1   2016-06-23
2   2015-03-15
3   2015-03-15
4   2015-03-15
Name: user_reg_date, dtype: datetime64[us]


____________________________________________________________________________________

**3. Data audit and imputing missing values using scaffolding**
____________________________________________________________________________________


In [13]:

# Convert columns to datetime and report values that could not be parsed
#Scaffolding only columns will missing and invalid datetime delivery_date and user_dob

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

    print(f"\n{column}:")
    print("Data type:", df[column].dtype)
    print("Sample values:")
    print(df[column].head())
    print("Missing/invalid dates:", df[column].isna().sum())

# Scaffold dates in column delivery_date using imputation (median) of order_date 

valid_delivery = df["delivery_date"].notna() & df["order_date"].notna()
median_delivery_days = ((df.loc[valid_delivery, "delivery_date"] -
     df.loc[valid_delivery, "order_date"]).dt.days.median())

if pd.notna(median_delivery_days):
    missing_delivery = df["delivery_date"].isna() & df["order_date"].notna()
    df.loc[missing_delivery, "delivery_date"] = (df.loc[missing_delivery, "order_date"]
        + pd.to_timedelta(median_delivery_days, unit="D"))

# Scaffold dates in column user_dob using imputation (median) of order_date 
# but only if age can be estimated from valid records.

today = pd.Timestamp.today().normalize()

dob = df["user_dob"]
birthday_not_yet_reached = (
    (dob.dt.month > today.month)
    | ((dob.dt.month == today.month) & (dob.dt.day > today.day)))

df["customer_age_years"] = (
    today.year - dob.dt.year - birthday_not_yet_reached.astype("Int64")).astype("Int64")
    
# Display corrupted and uparseable text to missing values

print("\nMissing dates after scaffolding:")
print(df[date_columns].isna().sum())


order_date:
Data type: datetime64[us]
Sample values:
0   2016-06-22
1   2016-06-22
2   2016-06-22
3   2016-06-22
4   2016-06-22
Name: order_date, dtype: datetime64[us]
Missing/invalid dates: 0

delivery_date:
Data type: datetime64[us]
Sample values:
0   2016-06-27
1          NaT
2   2016-06-27
3   2016-06-27
4   2016-06-27
Name: delivery_date, dtype: datetime64[us]
Missing/invalid dates: 9318

user_dob:
Data type: datetime64[us]
Sample values:
0   1969-04-17
1   1969-04-17
2   1970-04-22
3   1970-04-22
4   1970-04-22
Name: user_dob, dtype: datetime64[us]
Missing/invalid dates: 8725

user_reg_date:
Data type: datetime64[us]
Sample values:
0   2016-06-23
1   2016-06-23
2   2015-03-15
3   2015-03-15
4   2015-03-15
Name: user_reg_date, dtype: datetime64[us]
Missing/invalid dates: 0

Missing dates after scaffolding:
order_date          0
delivery_date       0
user_dob         8725
user_reg_date       0
dtype: int64


______________________________________________________________________________________________

**4. Feature creation, data cleaning and modification**
______________________________________________________________________________________________

In [14]:

# Ensure date columns are in datetime format

date_columns = ["order_date", "user_dob", "delivery_date"]
for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

# Create new feature by calculating age from order_date and user_dob

# 1. Feature creation (customer_age_years)

df["customer_age_years"] = ((df["order_date"] - df["user_dob"]).dt.days / 365.25)

# Filter out invalid or missing ages become blank

df.loc[~df["customer_age_years"].between(18, 150),"customer_age_years"] = pd.NA
df["customer_age_missing"] = df["customer_age_years"].isna().astype(int)

# 2. Feature creation (delivery_days)

df["delivery_days"] = (df["delivery_date"] - df["order_date"]).dt.days

# Negative or unavailable durations are invalid

df.loc[df["delivery_days"] < 0, "delivery_days"] = pd.NA

# 3. Feature creation (delivery_method)

# Delivery method: check under 7 days before under 14 days

df["Delivery_method"] = pd.NA
df.loc[df["delivery_days"] < 7, "Delivery_method"] = "Flight"
df.loc[df["delivery_days"].between(7, 13), "Delivery_method"] = "Road"
df.loc[df["delivery_days"] >= 14, "Delivery_method"] = "Shipping"

# 4. Feature creation (delivery_status)

# Delivery status based on delivery method

status_by_method = {
    "Road": "slight damage",
    "Flight": "no damage",
    "Shipping": "minor damage"}

df["Delivery_status"] = df["Delivery_method"].map(status_by_method)

# 5. Feature creation (Insurance_status)

# Insurance: price above 50 AND method is Flight or Shipping

df["Insurance_status"] = "not insured"
insured_condition = ((df["item_price"] > 50) & (df["Delivery_method"].isin(["Flight", "Shipping"])))

df.loc[insured_condition, "Insurance_status"] = "insured"

# Print outcomes on new columns (features) added to dataset

derived_columns = [
    "customer_age_years",
    "customer_age_missing",
    "delivery_days",
    "Delivery_method",
    "Delivery_status",
    "Insurance_status"]

print("New feature outcomes summary:")
print("=" * 60)

for column in derived_columns:
    print(f"\n{column}")
    print(f"  Missing/invalid values: {df[column].isna().sum()}")
    print(f"  Non-missing values:     {df[column].notna().sum()}")

    if column in [
        "customer_age_missing",
        "Delivery_method",
        "Delivery_status",
        "Insurance_status"]:
        
        print("  Value counts:")
        print(df[column].value_counts(dropna=False).to_string())
    else:
        print("  Summary:")
        print(df[column].describe().to_string())

print("\nSample rows:")
print(df[derived_columns].head(10).to_string(index=False))

New feature outcomes summary:

customer_age_years
  Missing/invalid values: 8767
  Non-missing values:     91233
  Summary:
count    91233.000000
mean        52.332795
std         11.270784
min         18.357290
25%         46.157426
50%         51.523614
75%         57.234771
max        115.805613

customer_age_missing
  Missing/invalid values: 0
  Non-missing values:     100000
  Value counts:
customer_age_missing
0    91233
1     8767

delivery_days
  Missing/invalid values: 1072
  Non-missing values:     98928
  Summary:
count    98928.000000
mean         6.987092
std         12.905671
min          0.000000
25%          2.000000
50%          3.000000
75%          4.000000
max        173.000000

Delivery_method
  Missing/invalid values: 1072
  Non-missing values:     98928
  Value counts:
Delivery_method
Flight      81383
Shipping    11815
Road         5730
<NA>         1072

Delivery_status
  Missing/invalid values: 1072
  Non-missing values:     98928
  Value counts:
Delivery_stat

In [15]:
# Inspecting the delivery_days missing/invalid results for cleaning

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["delivery_date"] = pd.to_datetime(df["delivery_date"], errors="coerce")

# Calculate delivery days
df["delivery_days"] = (df["delivery_date"] - df["order_date"]).dt.days

# Missing: either source date is missing/invalid, or the result is missing
missing_mask = (
    df["order_date"].isna()
    | df["delivery_date"].isna()
    | df["delivery_days"].isna())

# Invalid: negative delivery duration

negative_mask = df["delivery_days"].lt(0)

# Total invalid includes missing and negative values

invalid_mask = missing_mask | negative_mask

total_rows = len(df)
missing_count = int(missing_mask.sum())
negative_count = int(negative_mask.sum())
invalid_count = int(invalid_mask.sum())

print(f"Total rows: {total_rows}")
print(f"Missing delivery days: {missing_count}")
print(f"Negative delivery days: {negative_count}")
print(f"Total invalid/missing delivery days: {invalid_count}")

if total_rows:
    print(f"Negative days (% of all rows): {negative_count / total_rows * 100:.2f}%")

negative_mean = df.loc[negative_mask, "delivery_days"].mean()
print(f"Mean of negative delivery days: {negative_mean}")

print("\nFirst 20 rows with missing or invalid delivery days:")

columns_to_show = [col for col in ["order_id", "order_date", "delivery_date", "delivery_days"]
    if col in df.columns]

print(df.loc[invalid_mask, columns_to_show].head(20).to_string(index=False))

Total rows: 100000
Missing delivery days: 0
Negative delivery days: 1072
Total invalid/missing delivery days: 1072
Negative days (% of all rows): 1.07%
Mean of negative delivery days: -7877.41604477612

First 20 rows with missing or invalid delivery days:
order_date delivery_date  delivery_days
2016-06-23    1994-12-31          -7845
2016-06-23    1994-12-31          -7845
2016-06-23    1994-12-31          -7845
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-24    1994-12-31          -7846
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-25    1994-12-31          -7847
2016-06-26    1994-12-31          -7848
2016-06-26    1994-12-31          -7848
2016-06-26    1994-12-31

In [16]:
# Ensure date columns are in datetime format

date_columns = ["order_date", "user_dob", "delivery_date"]
for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

# 1. Create customer age and mark missing/invalid values

df["customer_age_years"] = ((df["order_date"] - df["user_dob"]).dt.days / 365.25)

df.loc[
    ~df["customer_age_years"].between(18, 150),
    "customer_age_years"] = pd.NA

df["customer_age_missing"] = df["customer_age_years"].isna().astype(int)

# Impute missing/invalid ages with the mean of valid ages

age_mean = df["customer_age_years"].mean()
df["customer_age_years"] = df["customer_age_years"].fillna(age_mean)

# 2. Identify negative delivery durations and adjust delivery dates

initial_delivery_days = (df["delivery_date"] - df["order_date"]).dt.days

negative_mask = initial_delivery_days < 0

print(f"Negative delivery durations to adjust: {negative_mask.sum()}")

# Add 7,860 days to delivery_date for rows with negative durations

df.loc[negative_mask, "delivery_date"] += pd.Timedelta(days=7860)

# Recalculate delivery days

df["delivery_days"] = (
    df["delivery_date"] - df["order_date"]).dt.days

# 3. Create delivery method

df["Delivery_method"] = pd.NA
df.loc[df["delivery_days"] < 7, "Delivery_method"] = "Flight"
df.loc[df["delivery_days"].between(7, 13), "Delivery_method"] = "Road"
df.loc[df["delivery_days"] >= 14, "Delivery_method"] = "Shipping"

# 4. Create delivery status

status_by_method = {
    "Road": "slight damage",
    "Flight": "no damage",
    "Shipping": "minor damage",}
df["Delivery_status"] = df["Delivery_method"].map(status_by_method)

# 5. Create insurance status

df["Insurance_status"] = "not insured"
insured_condition = (
    (df["item_price"] > 50)
    & df["Delivery_method"].isin(["Flight", "Shipping"]))
df.loc[insured_condition, "Insurance_status"] = "insured"

# Print outcomes for derived columns

derived_columns = [
    "customer_age_years",
    "customer_age_missing",
    "delivery_days",
    "Delivery_method",
    "Delivery_status",
    "Insurance_status",]

print("\nNew feature outcomes summary:")
print("=" * 60)

for column in derived_columns:
    print(f"\n{column}")
    print(f"  Missing values:     {df[column].isna().sum()}")
    print(f"  Non-missing values: {df[column].notna().sum()}")

    if column in [
        "customer_age_missing",
        "Delivery_method",
        "Delivery_status",
        "Insurance_status",]:
        print("  Value counts:")
        print(df[column].value_counts(dropna=False).to_string())
    else:
        print("  Summary:")
        print(df[column].describe().to_string())

print("\nSample rows:")
print(df[derived_columns].head(10).to_string(index=False))

Negative delivery durations to adjust: 1072

New feature outcomes summary:

customer_age_years
  Missing values:     0
  Non-missing values: 100000
  Summary:
count    100000.000000
mean         52.332795
std          10.765394
min          18.357290
25%          46.704997
50%          52.332795
75%          56.577687
max         115.805613

customer_age_missing
  Missing values:     0
  Non-missing values: 100000
  Value counts:
customer_age_missing
0    91233
1     8767

delivery_days
  Missing values:     0
  Non-missing values: 100000
  Summary:
count    100000.000000
mean          6.725490
std          13.312653
min         -65.000000
25%           2.000000
50%           3.000000
75%           4.000000
max         173.000000

Delivery_method
  Missing values:     0
  Non-missing values: 100000
  Value counts:
Delivery_method
Flight      82194
Shipping    11819
Road         5987

Delivery_status
  Missing values:     0
  Non-missing values: 100000
  Value counts:
Delivery_status
no

In [20]:
print(df.columns.tolist())

['order_item_id', 'order_date', 'delivery_date', 'item_id', 'item_size', 'item_color', 'brand_id', 'item_price', 'user_id', 'user_title', 'user_dob', 'user_state', 'user_reg_date', 'return', 'customer_age_years', 'customer_age_missing', 'delivery_days', 'Delivery_method', 'Delivery_status', 'Insurance_status']


____________________________________________________________________________________
**5. Preparing dataset for machine -learning model**
____________________________________________________________________________________

In [21]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 1. Select the required columns and print missing values in columns

model_columns = [
    "customer_age_years",
    "Delivery_method",       
    "Delivery_status",      
    "Insurance_status",
    "order_item_id",
    "item_id",
    "item_size",
    "item_color",
    "brand_id",
    "item_price",
    "user_state",
    "return"]

missing_columns = [col for col in model_columns if col not in df.columns]
if missing_columns:
    raise KeyError(
        f"Missing columns in df: {missing_columns}\n"
        f"Available columns: {df.columns.tolist()}")

model_df = df[model_columns].copy()

print("Selected model-training columns:")
print(model_df.columns.tolist())
print(f"\nNumber of rows before final preprocessing: {len(model_df):,}")

# 2. Target column should drop missing values and separate x (input features) and y (target column)

model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=["return"]).copy()

X = model_df.drop(columns=["return"])
y = model_df["return"].astype(int)


# 3. Define numerical and categorical features

numeric_features = [
    "customer_age_years",
    "order_item_id",
    "item_id",
    "brand_id",
    "item_price"]

categorical_features = [
    "Delivery_method",
    "Delivery_status",
    "Insurance_status",
    "item_size",
    "item_color",
    "user_state"]

# 4. Create preprocessing pipelines

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)])

# 5. Split datasets before fitting preprocessing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)


# 6. Fit on training data and transform both sets

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()


# 7. Convert processed arrays to DataFrames

X_train_final = pd.DataFrame(
    X_train_encoded,
    columns=feature_names,
    index=X_train.index).reset_index(drop=True)

X_test_final = pd.DataFrame(
    X_test_encoded,
    columns=feature_names,
    index=X_test.index).reset_index(drop=True)

y_train_final = y_train.reset_index(drop=True)
y_test_final = y_test.reset_index(drop=True)


# 8. Final checks of new dataset features

print("\n" + "=" * 70)
print("FINAL MODEL DATASET CHECK")
print("=" * 70)

print(f"\nRows retained:     {len(model_df):,}")
print(f"Training rows:       {len(X_train_final):,}")
print(f"Testing rows:        {len(X_test_final):,}")
print(f"Original features:   {X.shape[1]}")
print(f"Encoded features:    {X_train_final.shape[1]}")

print(f"\nTraining missing values: {X_train_final.isna().sum().sum()}")
print(f"Testing missing values:  {X_test_final.isna().sum().sum()}")

print("\nTarget distribution - training:")
print(y_train_final.value_counts().sort_index())

print("\nTarget distribution - testing:")
print(y_test_final.value_counts().sort_index())

print("\nFirst 5 rows of final training data:")
display(X_train_final.head())

print("\nFinal feature names:")
print(feature_names.tolist())

print("\nFinal training dataset shape:", X_train_final.shape)
print("Final testing dataset shape:", X_test_final.shape)

Selected model-training columns:
['customer_age_years', 'Delivery_method', 'Delivery_status', 'Insurance_status', 'order_item_id', 'item_id', 'item_size', 'item_color', 'brand_id', 'item_price', 'user_state', 'return']

Number of rows before final preprocessing: 100,000

FINAL MODEL-READY DATASET CHECK

Rows retained:       100,000
Training rows:       80,000
Testing rows:        20,000
Original features:   11
Encoded features:    206

Training missing values: 0
Testing missing values:  0

Target distribution - training:
return
0    43346
1    36654
Name: count, dtype: int64

Target distribution - testing:
return
0    10836
1     9164
Name: count, dtype: int64

First 5 rows of final training data:


,num__customer_age_years,num__order_item_id,num__item_id,num__brand_id,num__item_price,cat__Delivery_method_Flight,cat__Delivery_method_Road,cat__Delivery_method_Shipping,cat__Delivery_status_minor damage,cat__Delivery_status_no damage,...,cat__user_state_Hesse,cat__user_state_Lower Saxony,cat__user_state_Mecklenburg-Western Pomerania,cat__user_state_North Rhine-Westphalia,cat__user_state_Rhineland-Palatinate,cat__user_state_Saarland,cat__user_state_Saxony,cat__user_state_Saxony-Anhalt,cat__user_state_Schleswig-Holstein,cat__user_state_Thuringia
0,-0.282499,-1.585205,-1.080942,0.599255,0.522523,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.460339,1.166122,0.776069,-0.963462,-0.106733,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.001019,-1.346742,-0.633779,-0.466234,-0.001857,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.540761,-1.107136,-1.201990,-0.146587,-0.526238,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.474613,0.387724,1.109305,1.558195,0.732276,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0



Final feature names:
['num__customer_age_years', 'num__order_item_id', 'num__item_id', 'num__brand_id', 'num__item_price', 'cat__Delivery_method_Flight', 'cat__Delivery_method_Road', 'cat__Delivery_method_Shipping', 'cat__Delivery_status_minor damage', 'cat__Delivery_status_no damage', 'cat__Delivery_status_slight damage', 'cat__Insurance_status_insured', 'cat__Insurance_status_not insured', 'cat__item_size_1', 'cat__item_size_10', 'cat__item_size_10+', 'cat__item_size_100', 'cat__item_size_104', 'cat__item_size_105', 'cat__item_size_11', 'cat__item_size_11+', 'cat__item_size_116', 'cat__item_size_12', 'cat__item_size_12+', 'cat__item_size_128', 'cat__item_size_13', 'cat__item_size_14', 'cat__item_size_140', 'cat__item_size_152', 'cat__item_size_164', 'cat__item_size_176', 'cat__item_size_18', 'cat__item_size_19', 'cat__item_size_2', 'cat__item_size_2+', 'cat__item_size_20', 'cat__item_size_21', 'cat__item_size_22', 'cat__item_size_23', 'cat__item_size_24', 'cat__item_size_25', 'cat__

In [26]:
# Display new datasets column list and input features ready for model training 

print("Final input columns:")
print(model_df.columns.tolist())

print("\nFirst 10 rows:")
print(model_df.head(10))

Final input columns:
['customer_age_years', 'Delivery_method', 'Delivery_status', 'Insurance_status', 'order_item_id', 'item_id', 'item_size', 'item_color', 'brand_id', 'item_price', 'user_state', 'return']

First 10 rows:
   customer_age_years Delivery_method Delivery_status Insurance_status  \
0           47.181383          Flight       no damage      not insured   
1           47.181383          Flight       no damage      not insured   
2           46.168378          Flight       no damage          insured   
3           46.168378          Flight       no damage          insured   
4           46.168378          Flight       no damage      not insured   
5           46.168378          Flight       no damage      not insured   
6           46.168378          Flight       no damage          insured   
7           46.168378          Flight       no damage      not insured   
8           46.168378          Flight       no damage      not insured   
9           46.168378          Flight


                                                              **END**